# 🔎 ColabFold Search Tool Example

This notebook demonstrates how to search for homologous sequences using **ColabFold's sequence search**.

## 📖 What is ColabFold Search?

[ColabFold](https://github.com/sokrypton/ColabFold) provides fast multiple sequence alignment (MSA) generation by searching sequence databases. It's commonly used as a preprocessing step for structure prediction tools like AlphaFold2.

### Key Features:
- **Remote API** -- Search against ColabFold's servers (no local setup needed)
- **Local search** -- Run searches locally with MMseqs2
- **Metagenomic databases** -- Optional access to larger sequence databases
- **Fast & efficient** -- Optimized for speed compared to traditional methods

## 📥 Imports

## 📦 Shared Data Types

### `MSA`
A multiple sequence alignment result with alignment data and metadata.

| Field / Property | Type | Description |
|---|---|---|
| `aligned_sequences` | `List[str]` | The aligned sequences with gap characters (`-`) |
| `original_sequences` | `List[str]` | Original sequences without gap characters |
| `sequence_ids` | `List[str]` | Sequence identifiers |
| `alignment_length` | `int` | Number of columns in the alignment |
| `num_sequences` | `int` | Number of sequences in the alignment |
| `total_gaps` | `int` | Total number of gap characters in the alignment |
| `average_gap_fraction` | `float` | Average fraction of gaps across all sequences |

### `ColabfoldSearchQuery`
A single search query with sequence and optional metadata.

| Field | Type | Default | Description |
|---|---|---|---|
| `sequence` | `str` | *(required)* | Protein sequence to search for homologs |
| `sequence_id` | `Optional[str]` | `None` | Optional sequence identifier (auto-generated if not provided) |

In [1]:
from pathlib import Path

from proto_tools import (
    ColabfoldSearchConfig,
    ColabfoldSearchInput,
    run_colabfold_search,
)

## 🔍 1. Search for Homologs (Single Sequence)

Search for evolutionarily related sequences (homologs) in protein databases using a single protein sequence.

### API Reference

**`ColabfoldSearchInput`**

| Field | Type | Description |
|---|---|---|
| `queries` | `List[ColabfoldSearchQuery]` | List of protein sequences to search for homologs. Accepts raw strings, `(sequence, id)` tuples, or `ColabfoldSearchQuery` objects. |

**`ColabfoldSearchConfig`**

| Field | Type | Default | Description |
|---|---|---|---|
| `use_metagenomic_db` | `bool` | `False` | Whether to include metagenomic database in search |
| `msa_db_dir` | `str` | `"/large_storage/..."` | Path to local ColabFold/MMSeqs2 database directory |
| `database_name` | `str` | `"uniref30_2302_db"` | Name of the database to use |
| `sensitivity` | `Optional[float]` | `None` | MMseqs2 sensitivity parameter (`1.0`--`9.0`). Higher means more hits but slower search. |

**`ColabfoldSearchOutput`**

| Field | Type | Description |
|---|---|---|
| `results` | `List[ColabfoldSearchResult]` | List of MSA search results, one per input query |

**`ColabfoldSearchResult`**

| Field | Type | Description |
|---|---|---|
| `msa` | `Optional[MSA]` | Multiple Sequence Alignment containing homologous sequences, or `None` if no homologs found |
| `sequence_id` | `str` | Identifier for the searched sequence |
| `num_homologs_found` | `int` | Number of homologous sequences found (excludes the query sequence) |

In [2]:
# Hemoglobin subunit alpha (human) - first 60 residues
sequences = ["MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG"]
inputs = ColabfoldSearchInput(queries=sequences)

config = ColabfoldSearchConfig(
    search_mode="remote",  # Use the ColabFold API (no local setup needed)
    use_metagenomic_db=False,
)

result = run_colabfold_search(inputs, config)
print(f"Found {result.results[0].num_homologs_found} homologs.")

Found 3677 homologs.


## 🧬 2. Batch Search (Multiple Sequences)

Search for homologs of multiple sequences at once. You can optionally provide custom sequence IDs for easier tracking.

In [3]:
# Search multiple sequences with custom IDs using tuple format: (sequence, id)
queries = [
    ("MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG", "hemoglobin_alpha"),
    ("MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPK", "hemoglobin_beta"),
]

inputs = ColabfoldSearchInput(queries=queries)

config = ColabfoldSearchConfig(
    search_mode="remote",
    use_metagenomic_db=False,
)

batch_result = run_colabfold_search(inputs, config)

for res in batch_result.results:
    print(f"{res.sequence_id}: {res.num_homologs_found} homologs found")

hemoglobin_alpha: 3677 homologs found
hemoglobin_beta: 3956 homologs found


## 📊 3. Inspect MSA Results

Each result contains an `MSA` object with properties for exploring the alignment.

In [4]:
msa = batch_result.results[0].msa

print(f"Number of sequences: {msa.num_sequences}")
print(f"Alignment length:    {msa.alignment_length}")
print(f"Average gap fraction: {msa.average_gap_fraction:.3f}")
print(f"\nQuery sequence (first entry):")
print(f"  ID:  {msa.sequence_ids[0]}")
print(f"  Seq: {msa[0][:60]}...")

Number of sequences: 3678
Alignment length:    60
Average gap fraction: 0.063

Query sequence (first entry):
  ID:  101
  Seq: MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG...


## 💾 4. Export Results

Export MSA results to files for downstream analysis (e.g., structure prediction with AlphaFold2).

### Supported formats:
- **A3M** -- Compressed alignment format, used by ColabFold and AlphaFold2
- **FASTA** -- Standard sequence format, widely compatible

In [5]:
# Create output directory and export
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export results to A3M files (one per sequence)
batch_result.export("colabfold_a3m", export_path=output_dir, file_format="a3m")
print("A3M files exported to ./example_output/colabfold_a3m/")

# Export to FASTA format
batch_result.export("colabfold_fasta", export_path=output_dir, file_format="fasta")
print("FASTA files exported to ./example_output/colabfold_fasta/")

A3M files exported to ./example_output/colabfold_a3m/
FASTA files exported to ./example_output/colabfold_fasta/
